In [ ]:
import pandas as pd
from itertools import combinations
from collections import defaultdict, Counter


# ===== 1. Initial Setup and Data Loading =====

In [ ]:
# Import data from Excel files
try:
    student_raw = pd.read_excel("in/etudiant_without_out.xlsx",sheet_name="ne")
    print(student_raw.columns)
    print(student_raw.head())
except FileNotFoundError:
    print("Warning: etudiant.xlsx file not found. Student data will not be loaded.")
    student_raw = pd.DataFrame()

try:
    jury_raw = pd.read_excel("in/jury_new_down.xlsx")
    print(jury_raw.columns)
    print(jury_raw.head())
except FileNotFoundError:
    print("Error: jury.xlsx file not found. Cannot proceed without jury data.")
    raise

# ===== 2. Clean Jury Language Data =====

In [ ]:
jury_clean_language = jury_raw.copy()

def clean_language(row):
    options = row["Langue jury / jury language"].lower().split(";")
    if len(options) >= 3:
        row["Langue jury / jury language"] = "bilingue"
    else:
        row["Langue jury / jury language"] = options[0].lower()
    return row

jury_clean_language = jury_clean_language.apply(clean_language, axis=1)
jury_clean_language = jury_clean_language[jury_clean_language["type"].isin(["PA"])]
jury_clean_language.head()

# ===== 3. Extract Available Time Slots for Each Jury Member =====

In [ ]:
jury_slots = jury_clean_language.copy()
jours = ["lundi", "mardi", "mercredi", "jeudi", "vendredi"]

def extract_slots_with_days(row):
    slots = []
    for i, col in enumerate(jours):
        if pd.notna(row[col]):
            cleaned = str(row[col]).replace(" ", " ").replace(";", "").strip()
            for slot in cleaned.split():
                if slot:
                    jour = jours[i] if i < len(jours) else f"Jour{i + 1}"
                    slots.append((jour, slot))
    return tuple(sorted(set(slots)))


jury_slots["slots"] = jury_slots.apply(extract_slots_with_days, axis=1)
jury_slots.head()

In [ ]:
pma_teachers = []  # names of the second-priority teacher pool (redacted in this public copy)
print(len(pma_teachers))
def identifie_pma(row):
    if row["Prénom / First name"].lower()+" "+row["Nom / Name"].lower() in pma_teachers:
        row["type"] = "PAM"
    return row
jury_slots = jury_slots.apply(identifie_pma, axis=1)
print(jury_slots.columns)
jury_slots["type"].value_counts()

# ===== 4. IMPROVED JURY PAIRING ALGORITHM =====

In [ ]:
# Step 1: Count available slots per professor and analyze slot availability
# === Chargement et nettoyage ===
jury_slots['Prénom / First name'] = jury_slots['Prénom / First name'].str.lower().str.strip()
jury_slots['Nom / Name'] = jury_slots['Nom / Name'].str.lower().str.strip()
jury_slots['type'] = jury_slots['type'].str.upper().str.strip()
jury_slots['prof_name'] = jury_slots['Prénom / First name'] + " " + jury_slots['Nom / Name']
jury_slots['Heure de début'] = pd.to_datetime(jury_slots['Heure de début'], errors='coerce')

# === Scoring ===
type_points = {"PP": 3, "PAM": 2, "PA": 1}
jury_slots['score_base'] = jury_slots['type'].map(type_points).fillna(0)

# Bonus réactivité
jury_slots = jury_slots.sort_values("Heure de début")
total = len(jury_slots)
quart_25 = int(0.25 * total)
quart_50 = int(0.5 * total)

jury_slots['bonus_reactivité'] = 0
jury_slots.iloc[:quart_25, jury_slots.columns.get_loc('bonus_reactivité')] = 1
jury_slots.iloc[quart_25:quart_50, jury_slots.columns.get_loc('bonus_reactivité')] = 0.5

jury_slots['score_total'] = jury_slots['score_base'] + jury_slots['bonus_reactivité']

# === Création du dictionnaire de scores ===
prof_scores = jury_slots.set_index('prof_name')['score_total'].to_dict()

# === Comptage des disponibilités ===
prof_slot_counts = defaultdict(int)
slot_profs = defaultdict(list)

for _, row in jury_slots.iterrows():
    prof_name = row['prof_name']
    langue = row['Langue jury / jury language']
    for jour, slot in row['slots']:
        slot_key = (jour, slot)
        prof_slot_counts[prof_name] += 1
        slot_profs[slot_key].append({
            'name': prof_name,
            'langue': langue,
            'score': prof_scores.get(prof_name, 0)
        })

# === Création des jurys ===
jurys = []
assigned_profs_per_slot = defaultdict(set)

# Trier les slots par difficulté (peu de profs dispo)
#sorted_slots = sorted(slot_profs.items(), key=lambda x: len(x[1]))
capacity = 3
capcity_dict = {}
sorted_slots = slot_profs.items()
for (jour, horaire), profs in sorted_slots:
    capcity_dict[(jour, horaire)] = 0
    # Group professors by language
    lang_groups = defaultdict(list)
    for prof in profs:
        if prof['name'] not in assigned_profs_per_slot[(jour, horaire)]:
            lang_groups[prof['langue']].append(prof)

    # For each language group, pair professors prioritizing those with fewest slots
    for langue, lang_profs in lang_groups.items():
        # Sort professors by number of available slots (ascending)
        sorted_profs = sorted(lang_profs, key=lambda p: -p['score'])

        # Create pairs while professors are available
        i = 0
        while i + 1 < len(sorted_profs) - 1 and capcity_dict[(jour, horaire)] < capacity:
            prof1 = sorted_profs[i]
            prof2 = sorted_profs[i+1]

            # Create a jury with this pair
            jurys.append({
                "langue": langue,
                "slots": ((jour, horaire),),  # Single slot for this jury
                "members": [prof1['name'], prof2['name']],
            })
            capcity_dict[(jour, horaire)] += 1 
            # Mark these professors as assigned for this time slot
            assigned_profs_per_slot[(jour, horaire)].add(prof1['name'])
            assigned_profs_per_slot[(jour, horaire)].add(prof2['name'])

            i += 2
capcity_dict = {}
# Step 3: Create multi-slot juries for professors with identical availability patterns
# Group professors by their full availability pattern
profs_by_pattern = defaultdict(list)
for _, row in jury_slots.iterrows():
    prof_name = f"{row['Prénom / First name']} {row['Nom / Name']}"
    langue = row['Langue jury / jury language']

    profs_by_pattern[(langue, row['slots'])].append({
        'name': prof_name,
        'langue': langue
    })

# Create jurys for professors with matching patterns (considering who's already assigned)
for (langue, slots), profs in profs_by_pattern.items():
    # Skip if less than 2 professors with this pattern
    if len(profs) < 2:
        continue

    # Skip slot patterns with a single slot (already handled above)
    if len(slots) <= 1:
        continue

    # Filter out professors already assigned to these slots
    available_profs = []
    for prof in profs:
        already_assigned = False
        for jour, horaire in slots:
            if prof['name'] in assigned_profs_per_slot[(jour, horaire)]:
                already_assigned = True
                break
        if not already_assigned:
            available_profs.append(prof)

    # Create pairs from available professors
    i = 0
    while i < len(available_profs) - 1:
        prof1 = available_profs[i]
        prof2 = available_profs[i+1]

        # Create a jury with this pair for all their shared slots
        jurys.append({
            "langue": langue,
            "slots": slots,
            "members": [prof1['name'], prof2['name']],
        })

        # Mark these professors as assigned for all these time slots
        for jour, horaire in slots:
            assigned_profs_per_slot[(jour, horaire)].add(prof1['name'])
            assigned_profs_per_slot[(jour, horaire)].add(prof2['name'])

        i += 2

# Print some statistics
print(f"Total jurys created: {len(jurys)}")
slot_coverage = Counter()
for jury in jurys:
    for jour, horaire in jury["slots"]:
        slot_coverage[(jour, horaire)] += 1
print(f"Slots with jury coverage: {len(slot_coverage)}")
print(f"Total professors: {len(jury_slots)}")
print(f"Professors assigned to at least one jury: {len(set(name for jury in jurys for name in jury['members']))}")


# ===== 5. Create DataFrame of Jury Assignments =====

In [ ]:
jury_df = pd.DataFrame([{
    "Langue du jury": jury["langue"],
    "Créneaux": ", ".join([f"{jour} {heure}" for jour, heure in jury["slots"]]),
    "Membre 1": jury["members"][0],
    "Membre 2": jury["members"][1],
} for jury in jurys])

print(jury_df.shape[0])
jury_df.head()

# ===== 6. Clean and Normalize Professor Names =====

In [ ]:
def nettoyer_nom(nom):
    """Nettoie les noms pour les comparer plus facilement."""
    if pd.isna(nom):
        return ""
    return str(nom).strip().lower().replace("é", "e").replace("è", "e").replace("à", "a")

def normaliser_professeur(nom_complet):
    """
    Transforme un nom 'NOM Prénom' en 'prenom nom' (tout en minuscule, nettoyé)
    """
    if pd.isna(nom_complet):
        return ""

    parts = nom_complet.strip().split()
    if len(parts) >= 2:
        nom = parts[0].capitalize()
        prenom = " ".join(parts[1:]).capitalize()
        return f"{prenom} {nom}".lower()
    else:
        return nom_complet.strip().lower()

def parse_slots(creaneaux_str):
    """Découpe les créneaux sous forme (jour, horaire)."""
    slots = []
    if pd.isna(creaneaux_str):
        return []
    for entry in creaneaux_str.split(","):
        entry = entry.strip()
        if " " in entry:
            jour, horaire = entry.split(" ", 1)
            slots.append((jour.strip(), horaire.strip()))
    return slots

if not student_raw.empty:
    df_etudiants = student_raw.copy()
    df_jurys = jury_df.copy()
    df_etudiants["Professeur"] = df_etudiants["Professeur"].apply(normaliser_professeur)
    df_jurys["Membre 1"] = df_jurys["Membre 1"].apply(nettoyer_nom)
    df_jurys["Membre 2"] = df_jurys["Membre 2"].apply(nettoyer_nom)
    df_jurys["slots_jour_horaire"] = df_jurys["Créneaux"].apply(parse_slots)

    # Standardize language names
    df_etudiants = df_etudiants.replace("anglais", "english")
        # ===== 7. Prepare Jury Slots for Student Assignment =====
    jury_creneaux_disponibles = []
    for idx, row in df_jurys.iterrows():
        for jour, horaire in row["slots_jour_horaire"]:
            jury_creneaux_disponibles.append({
                "jury_id": idx,
                "langue": row["Langue du jury"],
                "jour": jour,
                "horaire": horaire,
                "membres": [row["Membre 1"], row["Membre 2"]],
                "disponible": True
            })
    print(f"Total jury slots available: {len(jury_creneaux_disponibles)}")
    
    # ===== 8. Group Students =====
    groupes_etudiants = []
    for langue in df_etudiants["Langue"].unique():
        etudiants_langue = df_etudiants[df_etudiants["Langue"] == langue].copy().reset_index(drop=True)
        for i in range(0, len(etudiants_langue), 6):
            groupe = etudiants_langue.iloc[i:i+6]
            if len(groupe) == 6:
                profs = set(groupe["Professeur"].apply(nettoyer_nom).unique())
                groupes_etudiants.append({
                    "langue": langue,
                    "etudiants": groupe,
                    "profs_preparateurs": profs
                })
    print(f"Total student groups: {len(groupes_etudiants)}")
        # ===== 9. Assign Student Groups to Juries =====
    assignments = []
    logs = []

    for idx_groupe, groupe in enumerate(groupes_etudiants):
        assigned = False
        langue_groupe = groupe["langue"]

        logs.append(f"\nGroupe #{idx_groupe + 1} — Langue: {langue_groupe}")
        logs.append(f"   Professeurs préparateurs: {groupe['profs_preparateurs']}")
        logs.append("   Étudiants dans le groupe :")
        for _, etu in groupe["etudiants"].iterrows():
            logs.append(f"     - {etu['Prenom']} {etu['Nom']} ({etu['MailEtudiant']})")

        # Filtrage des jurys compatibles (par langue ou bilingue)
        if langue_groupe == "english":
            jurys_possibles = [
                j for j in jury_creneaux_disponibles
                if j["disponible"] and (j["langue"] == "english" or j["langue"] == "bilingue")
            ]
            # Priorité : english > bilingue
            jurys_possibles.sort(key=lambda j: 0 if j["langue"] == "english" else 1)

        elif langue_groupe == "français":
            jurys_possibles = [
                j for j in jury_creneaux_disponibles
                if j["disponible"] and (j["langue"] == "français" or j["langue"] == "bilingue")
            ]
            # Priorité : français > bilingue
            jurys_possibles.sort(key=lambda j: 0 if j["langue"] == "français" else 1)

        else:
            jurys_possibles = []

        # Parcourir les jurys filtrés
        for slot in jurys_possibles:
            logs.append(f"   Tentative avec jury {slot['membres']} — {slot['jour']} {slot['horaire']} — Langue: {slot['langue']}")

            jury_profs = set(slot["membres"])
            conflits = groupe["profs_preparateurs"].intersection(jury_profs)

            if conflits:
                logs.append(f"     Conflit avec professeurs du jury : {conflits}")
                continue

            # Affectation OK
            logs.append("     Jury attribué à ce groupe.")
            for _, etu in groupe["etudiants"].iterrows():
                assignments.append({
                    "Nom": etu["Nom"],
                    "Prenom": etu["Prenom"],
                    "Langue": etu["Langue"],
                    "MailEtudiant": etu["MailEtudiant"],
                    "Professeur de préparation": etu["Professeur"],
                    "Jury_Membre_1": slot["membres"][0],
                    "Jury_Membre_2": slot["membres"][1],
                    "Jour": slot["jour"],
                    "Horaire": slot["horaire"]
                })
            slot["disponible"] = False
            assigned = True
            break

        if not assigned:
            logs.append("   Aucun jury compatible trouvé pour ce groupe.")
             # ===== 10. Export Results =====
    if assignments:
        df_affectation = pd.DataFrame(assignments)
        df_affectation.to_excel("affectation_etudiants_jurys_par_creneaux.xlsx", index=False)
        print(f"Successfully assigned {len(assignments)/6:.0f} student groups to juries")
    else:
        print("No student assignments generated")

    # Logs (groupes non assignés)
    with open("logs_non_assignes.txt", "w", encoding="utf-8") as f:
        for log in logs:
            f.write(log + "\n")

    print("Fichiers générés : affectation_etudiants_jurys_par_creneaux.xlsx et logs_non_assignes.txt")
else:
    # If no student data, just export the jury pairings
    jury_df.to_excel("jury_pairings.xlsx", index=False)
    print("Fichier généré : jury_pairings.xlsx (student assignments skipped)")


In [ ]:
import pandas as pd

# Charger le fichier d’affectation généré
df_affectation = pd.read_excel("affectation_etudiants_jurys_par_creneaux.xlsx")

# Extraire les colonnes des membres de jury
jury_membres = pd.concat([
    df_affectation["Jury_Membre_1"],
    df_affectation["Jury_Membre_2"]
])

# Nettoyage des noms (optionnel si déjà propre)
jury_membres = jury_membres.dropna().str.lower().str.strip()

# Obtenir la liste unique
profs_utilises = sorted(jury_membres.unique())

# Afficher le résultat
print(f"{len(profs_utilises)} professeurs ont été affectés à au moins un jury :\n")
for nom in profs_utilises:
    print("-", nom)
